## Add required fields Tasks for point, line, polygon layers

This notebook updates the schema of an existing point, line or polygon feature layer, preparing it to support the Tasks functionality in ArcGIS Field Maps.

The notebook performs the following actions:

- Adds a series of fields and necessary domains
- Enables attachments if needed 

Note: you must be the owner of the feature layer or an organization admin user

### Steps

0. Identify the target hosted feature layer that you want to enable with tasks. Editor tracking and sync should be enabled.

1. If necessary, update the following variables directly in the cell
      - `ASSIGN_TABLE_INDEX`

2. Run through the cells. Fill out any required inputs.


**Reference:**
Tasks Information Model: https://doc.arcgis.com/en/field-maps/latest/prepare-maps/prepare-tasks.htm#ESRI_SECTION1_3BBDB65A5B75485D89FC4BE0F92A32BC

## 1 — Import libraries

In [ ]:
# Import libraries
import arcgis
from arcgis.gis import GIS
from arcgis.features import FeatureLayerCollection
import uuid

## 2 — Sign in

In [ ]:
# Sign in to organization
portal_url = input('Enter ArcGIS Online or Enterprise URL: ') # (e.g., https://www.arcgis.com, https://YOURORG.maps.arcgis.com)
username   = input('Enter username: ')

gis = GIS(portal_url, username)
# gis = GIS('home') # use this method when running as a hosted notebook in AGOL/Enterprise

print(f'Connected as: {gis.properties.user.username}')

## 3 — Specify target feature layer info

In [ ]:
# Specify target feature layer and layer index
item_id = input('Enter item ID of target feature layer: ') # (e.g., 4rts7307f134432ah384rt5e8dnh838c)
feature_layer_index = int(input('Enter numeric index of target layer: ')) # (e.g., 0)

item = gis.content.get(item_id)

if item is None:
    raise TypeError('Cannot find item')

if item.type != 'Feature Service':
    raise TypeError('Item is not a Feature Service')

feature_layer = FeatureLayerCollection.fromitem(item).layers[feature_layer_index]

VALID_GEOMETRY_TYPES = {
    'esriGeometryPoint',
    'esriGeometryPolyline',
    'esriGeometryPolygon',
}

if feature_layer.properties.geometryType not in VALID_GEOMETRY_TYPES:
    raise TypeError('Feature layer must be a point, polyline, or polygon layer')

**Workforce Assigment Types (Optional)**
If your task layer originated from a workforce project, especially if you ran the **Flatten Workforce Service** notebook to create it, you may want to bring your Assignment Type values domain from Workforce and apply them to the domain of the `esritask_type` field. Use the next cell to specify your Workforce feature service ID and create the function to gather the domain values.

In [ ]:
# Specify Workforce feature service
workforce_fs_id = input('Enter item ID of the Workforce Assignment table: ')
ASSIGN_TABLE_INDEX = 1  # Index of the assignment type table

wf_item = gis.content.get(workforce_fs_id)

if wf_item is None:
    raise TypeError('Cannot Find item')

flc = FeatureLayerCollection.fromitem(wf_item)

# Check the index of the assignment type table
if flc.tables[ASSIGN_TABLE_INDEX].properties.name != 'Assignment Types':
    raise TypeError('Please enter the table index of your assignment type table')

# Query the assignment type table for unique descriptions
assignment_types_query = flc.tables[ASSIGN_TABLE_INDEX].query(
    out_fields = 'description', return_distinct_values=True
)

assignment_types = [f.attributes['description'] for f in assignment_types_query]

if not assignment_types:
    print('No assignment types found')
else:
    print('All assignment types from Workforce project:')
    print(assignment_types)

## 4 — Define the required Task fields

In [ ]:
# Store either the Workforce feature service-based domains, or a default set of coded value domains
try:
    domain_codes = [{'name': name, 'code': code} for code, name in enumerate(assignment_types)]
except NameError:
    domain_codes = [
        {'name': 'Python Added Fields 1', 'code': 0},  # You can update these domains to your desired values as well.
        {'name': 'Python Added Fields 2', 'code': 1},
    ]

# Get existing fields from feature service as a lowercase set for easy comparison
feature_layer_fields = {
    field['name'].lower() for field in feature_layer.properties.fields
}

# Define all required task fields
required_fields = []

if 'esritask_type' not in feature_layer_fields:
    required_fields.append(
        {
            'name': 'esritask_type',
            'type': 'esriFieldTypeInteger',
            'alias': 'Task Type',
            'sqlType': 'sqlTypeOther',
            'nullable': True,
            'editable': True,
            'domain': {
                'type': 'codedValue',
                'name': 'ESRITASK_TYPE_DOMAIN_' + str(uuid.uuid4()),
                'codedValues': domain_codes,
            },
            'defaultValue': 0,
        }
    )

if 'esritask_status' not in feature_layer_fields:
    required_fields.append(
        {
            'name': 'esritask_status',
            'type': 'esriFieldTypeInteger',
            'alias': 'Status',
            'sqlType': 'sqlTypeOther',
            'nullable': True,
            'editable': True,
            'domain': {
                'type': 'codedValue',
                'name': 'ESRITASK_STATUS_DOMAIN_' + str(uuid.uuid4()),
                'codedValues': [
                    {'name': 'Unassigned', 'code': 0},
                    {'name': 'Assigned', 'code': 1},
                    {'name': 'In Progress', 'code': 2},
                    {'name': 'Completed', 'code': 3},
                ],
            },
            'defaultValue': 0,
        }
    )

if 'esritask_assignee' not in feature_layer_fields:
    required_fields.append(
        {
            'name': 'esritask_assignee',
            'type': 'esriFieldTypeString',
            'alias': 'Assignee',
            'sqlType': 'sqlTypeOther',
            'length': 255,
            'nullable': True,
            'editable': True,
            'domain': {
                'type': 'codedValue',
                'name': 'ESRITASK_ASSIGNEE_DOMAIN_' + str(uuid.uuid4()),
                'codedValues': [
                    {'name': 'Assignee name 1', 'code': 'assignee_username_1'},
                    {'name': 'Assignee name 2', 'code': 'assignee_username_2'},
                ],
            },
            'defaultValue': None,
        }
    )

if 'esritask_priority' not in feature_layer_fields:
    required_fields.append(
        {
            'name': 'esritask_priority',
            'type': 'esriFieldTypeInteger',
            'alias': 'Priority',
            'sqlType': 'sqlTypeOther',
            'nullable': True,
            'editable': True,
            'domain': {
                'type': 'codedValue',
                'name': 'ESRITASK_PRIORITY_DOMAIN_' + str(uuid.uuid4()),
                'codedValues': [
                    {'name': 'None', 'code': 0},
                    {'name': 'Low', 'code': 1},
                    {'name': 'Medium', 'code': 2},
                    {'name': 'High', 'code': 3},
                    {'name': 'Critical', 'code': 4},
                ],
            },
            'defaultValue': 0,
        }
    )

if 'esritask_duedate' not in feature_layer_fields:
    required_fields.append(
        {
            'name': 'esritask_duedate',
            'type': 'esriFieldTypeDate',
            'alias': 'Due Date',
            'sqlType': 'sqlTypeOther',
            'nullable': True,
            'editable': True,
        }
    )

if 'esritask_description' not in feature_layer_fields:
    required_fields.append(
        {
            'name': 'esritask_description',
            'type': 'esriFieldTypeString',
            'alias': 'Description',
            'sqlType': 'sqlTypeOther',
            'nullable': True,
            'editable': True,
            'length': 4000,
        }
    )

if 'esritask_notes' not in feature_layer_fields:
    required_fields.append(
        {
            'name': 'esritask_notes',
            'type': 'esriFieldTypeString',
            'alias': 'Notes',
            'sqlType': 'sqlTypeOther',
            'nullable': True,
            'editable': True,
            'length': 4000,
        }
    )

if 'globalid' not in feature_layer_fields:
    required_fields.append(
        {
            'name': 'GlobalID',
            'type': 'esriFieldTypeGlobalID',
            'alias': 'GlobalID',
            'sqlType': 'sqlTypeOther',
            'nullable': False,
            'editable': False,
            'defaultValue': 'NEWID() WITH VALUES',
        }
    )

print(required_fields)

## 5 — Add missing fields to the layer definition

In [ ]:
# Add missing fields to the layer definition
if not required_fields:
    print('No missing task fields. No changes made.')
else:
    response = feature_layer.manager.add_to_definition({'fields': required_fields})
    if response.get('success'):
        print(f'Successfully added {len(required_fields)} task field(s).')
    else:
        print('Failed to update feature layer service definition.')
        print(response)